In [ ]:
# Reserved source-cell index.


In [ ]:
# Reserved source-cell index.


In [ ]:
# Reserved source-cell index.


In [ ]:
from pathlib import Path
import os
import sys
_study_root = Path(os.environ["PRESCRIPTION_OUTPUT_ROOT"]).resolve().parents[1]
_runtime_root = _study_root / "runtime"
sys.path.insert(0, str(_runtime_root / "src"))


In [ ]:
# Reserved source-cell index.


In [ ]:
from pathlib import Path
import json
import math
import os
import platform
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.signal import find_peaks

from hat_revision_pipeline.gorkov_core import (
    AcousticMedium,
    ArrayGeometry,
    SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
    transfer_matrix,
)
from hat_revision_pipeline.sota import (
    VortexControlSpec,
    iterative_back_projection,
    phase_only_signature_projection_audit,
    vortex_control_points,
)

FREQUENCY_HZ = 40_000.0
MEDIUM = AcousticMedium()
WAVELENGTH_M = MEDIUM.sound_speed_m_s / FREQUENCY_HZ
CONTROL_POINTS = 8
NOMINAL_RADIUS_LAMBDA = 1.40
REFERENCE_SEPARATION_LAMBDA = 26.7
OPPOSED_SEPARATIONS_LAMBDA = np.asarray((26.7, 30.0, 35.0, 40.0, 45.0), dtype=float)
OPPOSED_RADIUS_CANDIDATES_LAMBDA = np.round(np.arange(0.80, 2.201, 0.05), 2)
SINGLE_RADIUS_CANDIDATES_LAMBDA = np.asarray(
    (0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 0.90, 1.10, 1.40), dtype=float
)
SINGLE_RETAINED_RADIUS_LAMBDA = 0.45
SINGLE_TARGET_M = np.asarray((0.0, 0.0, 0.050), dtype=float)
SINGLE_GEOMETRY = ArrayGeometry.square(side=16, pitch_m=0.010)

OUTPUT_DIR = Path(os.environ.get(
    "PRESCRIPTION_OUTPUT_ROOT", str(Path.cwd() / "prescription_outputs")
)).resolve()
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "data"
for directory in (FIGURE_DIR, TABLE_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 9.0,
    "axes.labelsize": 8.5,
    "legend.fontsize": 7.4,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Frequency: {FREQUENCY_HZ / 1000:.1f} kHz")
print(f"Wavelength: {1e3 * WAVELENGTH_M:.3f} mm")
print(f"Output directory: {OUTPUT_DIR}")



In [ ]:
def paired_arrays(separation_lambda: float) -> tuple[ArrayGeometry, ArrayGeometry]:
    separation_m = float(separation_lambda) * WAVELENGTH_M
    if not np.isfinite(separation_m) or separation_m <= 0.0:
        raise ValueError("separation_lambda must be finite and positive")
    lower = ArrayGeometry.square(
        16, 0.010, z_m=-separation_m / 2.0, normal=(0.0, 0.0, 1.0)
    )
    upper = ArrayGeometry.square(
        16, 0.010, z_m=+separation_m / 2.0, normal=(0.0, 0.0, -1.0)
    )
    return lower, upper


def paired_transfer(points_m: np.ndarray, arrays: tuple[ArrayGeometry, ArrayGeometry]) -> np.ndarray:
    lower, upper = arrays
    return (
        transfer_matrix(
            points_m, lower, FREQUENCY_HZ,
            source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
        )
        + transfer_matrix(
            points_m, upper, FREQUENCY_HZ,
            source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
        )
    )


def synthesize_from_transfer(
    method: str,
    transfer: np.ndarray,
    signature: np.ndarray,
) -> dict:
    started = time.perf_counter()
    if method == "IB":
        result = iterative_back_projection(
            transfer,
            signature,
            maxiter=200,
            tolerance_rad=0.01,
            control_group_size=CONTROL_POINTS,
        )
    elif method == "GS":
        result = phase_only_signature_projection_audit(
            transfer,
            signature,
            iterations=100,
            control_group_size=CONTROL_POINTS,
        )
    else:
        raise ValueError("method must be 'IB' or 'GS'")
    return {
        "method": method,
        "phase_rad": np.asarray(result.phase_rad, dtype=float),
        "iterations": int(result.iterations),
        "converged": result.converged,
        "terminal_projective_change_rad": result.terminal_projective_change_rad,
        "synthesis_wall_s": float(time.perf_counter() - started),
    }


def paired_command(method: str, separation_lambda: float, radius_lambda: float) -> dict:
    arrays = paired_arrays(separation_lambda)
    controls, signature = vortex_control_points(
        np.zeros(3),
        WAVELENGTH_M,
        VortexControlSpec(CONTROL_POINTS, float(radius_lambda), 1),
    )
    control_transfer = paired_transfer(controls, arrays).reshape(CONTROL_POINTS, 256)
    command = synthesize_from_transfer(method, control_transfer, signature)
    command.update({
        "arrays": arrays,
        "separation_lambda": float(separation_lambda),
        "radius_lambda": float(radius_lambda),
        "transfer": control_transfer,
        "signature": np.asarray(signature, dtype=np.complex128),
    })
    return command


def single_command(method: str, radius_lambda: float) -> dict:
    controls, signature = vortex_control_points(
        SINGLE_TARGET_M,
        WAVELENGTH_M,
        VortexControlSpec(CONTROL_POINTS, float(radius_lambda), 1),
    )
    control_transfer = transfer_matrix(
        controls,
        SINGLE_GEOMETRY,
        FREQUENCY_HZ,
        source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
    ).reshape(CONTROL_POINTS, 256)
    command = synthesize_from_transfer(method, control_transfer, signature)
    command.update({
        "geometry": SINGLE_GEOMETRY,
        "target_m": SINGLE_TARGET_M.copy(),
        "radius_lambda": float(radius_lambda),
        "transfer": control_transfer,
        "signature": np.asarray(signature, dtype=np.complex128),
    })
    return command


def pressure_from_transfer(
    phase_rad: np.ndarray,
    points_m: np.ndarray,
    transfer_builder,
    *,
    chunk_points: int = 512,
) -> np.ndarray:
    points = np.asarray(points_m, dtype=float)
    original_shape = points.shape[:-1]
    flat = points.reshape(-1, 3)
    values = np.empty(len(flat), dtype=np.complex128)
    drive = np.exp(1j * np.asarray(phase_rad, dtype=float))
    for start in range(0, len(flat), int(chunk_points)):
        stop = min(start + int(chunk_points), len(flat))
        transfer = transfer_builder(flat[start:stop]).reshape(stop - start, len(drive))
        values[start:stop] = transfer @ drive
    return values.reshape(original_shape)


def single_first_peak_profile(
    phase_rad: np.ndarray,
    radii_m: np.ndarray,
    *,
    angular_samples: int,
) -> tuple[np.ndarray, int]:
    theta = np.linspace(0.0, 2.0 * np.pi, int(angular_samples), endpoint=False)
    rr, tt = np.meshgrid(np.asarray(radii_m), theta, indexing="ij")
    points = SINGLE_TARGET_M[None, None, :] + np.stack(
        (rr * np.cos(tt), rr * np.sin(tt), np.zeros_like(rr)), axis=-1
    )
    amplitude = np.abs(pressure_from_transfer(
        phase_rad,
        points,
        lambda value: transfer_matrix(
            value, SINGLE_GEOMETRY, FREQUENCY_HZ,
            source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
        ),
    )).mean(axis=1)
    peaks, _ = find_peaks(
        amplitude,
        prominence=max(float(np.max(amplitude)) * 0.01, 1.0e-30),
        distance=4,
    )
    peaks = peaks[np.asarray(radii_m)[peaks] >= 0.00015]
    peak_index = int(peaks[0]) if len(peaks) else int(np.argmax(amplitude))
    return amplitude / max(float(amplitude[peak_index]), 1.0e-300), peak_index


def single_map(
    phase_rad: np.ndarray,
    *,
    half_width_m: float = 0.012,
    samples: int = 151,
    normalize: bool = True,
):
    axis = np.linspace(-half_width_m, half_width_m, int(samples))
    xx, yy = np.meshgrid(axis, axis, indexing="xy")
    points = SINGLE_TARGET_M[None, None, :] + np.stack((xx, yy, np.zeros_like(xx)), axis=-1)
    amplitude = np.abs(pressure_from_transfer(
        phase_rad,
        points,
        lambda value: transfer_matrix(
            value, SINGLE_GEOMETRY, FREQUENCY_HZ,
            source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
        ),
    ))
    if normalize:
        amplitude /= max(float(np.max(amplitude)), 1.0e-300)
    return axis, amplitude


def panel_label(ax: plt.Axes, label: str) -> None:
    ax.text(-0.13, 1.05, f"({label})", transform=ax.transAxes, weight="bold", va="bottom")


def save_figure(fig: plt.Figure, stem: str) -> Path:
    path = FIGURE_DIR / f"{stem}.png"
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", facecolor="white")
    payload = buffer.getvalue()
    path.write_bytes(payload)
    plt.close(fig)
    print(f"Saved {path}")
    return path


In [ ]:
# The paired implementation and explicit 512-source implementation are identical at the midpoint.
guard_arrays = paired_arrays(REFERENCE_SEPARATION_LAMBDA)
guard_controls, guard_signature = vortex_control_points(
    np.zeros(3), WAVELENGTH_M, VortexControlSpec(CONTROL_POINTS, NOMINAL_RADIUS_LAMBDA, 1)
)
guard_lower = transfer_matrix(
    guard_controls, guard_arrays[0], FREQUENCY_HZ,
    source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
).reshape(CONTROL_POINTS, 256)
guard_upper = transfer_matrix(
    guard_controls, guard_arrays[1], FREQUENCY_HZ,
    source_scale_pa_m=SOURCE_SCALE_PA_M_PER_MURATA_UNIT,
).reshape(CONTROL_POINTS, 256)
assert float(np.max(np.abs(guard_lower - guard_upper))) <= 1.0e-12
guard_paired = paired_command("GS", REFERENCE_SEPARATION_LAMBDA, NOMINAL_RADIUS_LAMBDA)
print(f"Midpoint paired-transfer difference: {np.max(np.abs(guard_lower - guard_upper)):.3e}")
print(f"Independent phase coordinates: {len(guard_paired['phase_rad'])}")


In [ ]:
# Reserved source-cell index.


In [ ]:
opposed_started = time.perf_counter()
opposed_radial_axis_m = np.linspace(0.0, 0.018, 181)
opposed_theta_rad = np.linspace(0.0, 2.0 * np.pi, 48, endpoint=False)
opposed_rr, opposed_tt = np.meshgrid(opposed_radial_axis_m, opposed_theta_rad, indexing="ij")
opposed_profile_points_m = np.stack(
    (opposed_rr * np.cos(opposed_tt), opposed_rr * np.sin(opposed_tt), np.zeros_like(opposed_rr)),
    axis=-1,
).reshape(-1, 3)
nominal_index = int(np.where(OPPOSED_RADIUS_CANDIDATES_LAMBDA == NOMINAL_RADIUS_LAMBDA)[0][0])

OPPOSED_COMMANDS = {}
OPPOSED_PROFILES = {}
profile_rows = []
selection_rows = []

for method in ("IB", "GS"):
    profiles_by_separation = {}
    for separation_lambda in OPPOSED_SEPARATIONS_LAMBDA:
        arrays = paired_arrays(float(separation_lambda))
        profile_transfer = paired_transfer(opposed_profile_points_m, arrays).reshape(-1, 256)
        phases = []
        synthesis_times = []
        for radius_lambda in OPPOSED_RADIUS_CANDIDATES_LAMBDA:
            command = paired_command(method, float(separation_lambda), float(radius_lambda))
            key = (float(separation_lambda), method, float(radius_lambda))
            OPPOSED_COMMANDS[key] = command
            phases.append(command["phase_rad"])
            synthesis_times.append(command["synthesis_wall_s"])
        amplitude = np.abs(
            profile_transfer @ np.exp(1j * np.asarray(phases).T)
        ).reshape(181, 48, len(OPPOSED_RADIUS_CANDIDATES_LAMBDA)).mean(axis=1)
        profiles = amplitude / np.maximum(amplitude.max(axis=0, keepdims=True), 1.0e-30)
        profiles_by_separation[float(separation_lambda)] = profiles
        OPPOSED_PROFILES[(float(separation_lambda), method)] = profiles
        for candidate_index, radius_lambda in enumerate(OPPOSED_RADIUS_CANDIDATES_LAMBDA):
            profile_rows.append({
                "array_separation_lambda": float(separation_lambda),
                "array_separation_mm": float(1e3 * separation_lambda * WAVELENGTH_M),
                "target_z_m": 0.0,
                "method": method,
                "control_points": CONTROL_POINTS,
                "radius_lambda": float(radius_lambda),
                "synthesis_wall_s": float(synthesis_times[candidate_index]),
                "radial_window_mm": 18.0,
                "radial_samples": 181,
                "angular_samples": 48,
                "normalization": "each profile divided by its global maximum over 0--18 mm",
            })

    reference_profile = profiles_by_separation[REFERENCE_SEPARATION_LAMBDA][:, nominal_index]
    for separation_lambda in OPPOSED_SEPARATIONS_LAMBDA:
        profiles = profiles_by_separation[float(separation_lambda)]
        rmse = np.sqrt(np.mean((profiles - reference_profile[:, None]) ** 2, axis=0))
        selected_index = int(np.argmin(rmse))
        selected_radius = float(OPPOSED_RADIUS_CANDIDATES_LAMBDA[selected_index])
        for row in profile_rows:
            if row["method"] == method and np.isclose(
                row["array_separation_lambda"], separation_lambda
            ):
                candidate_index = int(np.where(
                    OPPOSED_RADIUS_CANDIDATES_LAMBDA == row["radius_lambda"]
                )[0][0])
                row["radial_profile_rmse_to_method_reference"] = float(rmse[candidate_index])
                row["selected"] = bool(candidate_index == selected_index)
                row["fixed_1p4"] = bool(candidate_index == nominal_index)
                row["reference"] = (
                    f"same-method D/lambda={REFERENCE_SEPARATION_LAMBDA:.1f}, R/lambda=1.40"
                )
        selection_rows.append({
            "array_separation_lambda": float(separation_lambda),
            "array_separation_mm": float(1e3 * separation_lambda * WAVELENGTH_M),
            "target_z_m": 0.0,
            "method": method,
            "selected_radius_lambda": selected_radius,
            "selected_profile_rmse": float(rmse[selected_index]),
            "fixed_1p4_profile_rmse": float(rmse[nominal_index]),
            "rmse_reduction": float(rmse[nominal_index] - rmse[selected_index]),
            "candidate_count": len(OPPOSED_RADIUS_CANDIDATES_LAMBDA),
            "selection_rule": "discrete argmin of normalized radial-profile RMSE",
        })

OPPOSED_RADIUS_AUDIT = pd.DataFrame(profile_rows)
OPPOSED_RADIUS_SELECTION = pd.DataFrame(selection_rows)
OPPOSED_RADIUS_AUDIT.to_csv(TABLE_DIR / "opposed_array_radius_transfer_audit.csv", index=False)
OPPOSED_RADIUS_SELECTION.to_csv(TABLE_DIR / "opposed_array_radius_selection_summary.csv", index=False)
print(OPPOSED_RADIUS_SELECTION.to_string(index=False))
print(f"Opposed-array audit wall time: {time.perf_counter() - opposed_started:.3f} s")


In [ ]:
# Reserved source-cell index.


In [ ]:
# Current compact FE command for the established single-sided radius diagnostic.
from hat_revision_pipeline import pipeline as _pipeline
from hat_revision_pipeline.sota import solve_corrected_gorkov_fe
from hat_revision_pipeline.gorkov_core import Condition, SingleTargetObjective
from hat_revision_pipeline.fixed_fe_endpoints import fixed_single_fe_method_spec
from hat_revision_pipeline.fe_solver_policy import FE_SOLVER_REVISION
from hat_revision_pipeline.cache import CacheStore, digest_array
_reference_spec = fixed_single_fe_method_spec()
_reference_objective = SingleTargetObjective(Condition(
    "Appendix E FE reference", SINGLE_GEOMETRY.positions_m, tuple(SINGLE_TARGET_M),
    FREQUENCY_HZ, _reference_spec.curvature_weight), _reference_spec)
_reference_initial = np.random.default_rng(172817509).uniform(-np.pi, np.pi, len(SINGLE_GEOMETRY.positions_m))
_reference_cache = CacheStore(_study_root / "cache", namespace="appendix-E-reference")
_reference_run, _, _ = _reference_cache.get_or_compute("fixed_single_fe_endpoint", dict(
    fe_solver_policy=FE_SOLVER_REVISION, seed=172817509,
    target_m=SINGLE_TARGET_M.tolist(), initial_phase=digest_array(_reference_initial),
    alpha_per_m=_reference_spec.alpha_per_m, curvature_weight=np.asarray(_reference_spec.curvature_weight).tolist(), maxiter=10000, gtol=1e-8),
    lambda: solve_corrected_gorkov_fe(_reference_objective, _reference_initial, maxiter=10000, gtol=1e-8))
single_fe_phase = np.asarray(_reference_run.phase_rad)

single_radial_axis_m = np.linspace(0.0, 0.012, 161)
single_fe_profile, single_fe_peak_index = single_first_peak_profile(
    single_fe_phase, single_radial_axis_m, angular_samples=96
)
SINGLE_COMMANDS = {}
single_rows = []
for method in ("IB", "GS"):
    for radius_lambda in SINGLE_RADIUS_CANDIDATES_LAMBDA:
        command = single_command(method, float(radius_lambda))
        SINGLE_COMMANDS[(method, float(radius_lambda))] = command
        profile, peak_index = single_first_peak_profile(
            command["phase_rad"], single_radial_axis_m, angular_samples=96
        )
        single_rows.append({
            "method": method,
            "control_points": CONTROL_POINTS,
            "radius_lambda": float(radius_lambda),
            "radial_profile_rmse_to_deposited_fe": float(np.sqrt(np.mean(
                (profile - single_fe_profile) ** 2
            ))),
            "realized_first_maximum_mm": float(1e3 * single_radial_axis_m[peak_index]),
            "realized_first_maximum_lambda": float(single_radial_axis_m[peak_index] / WAVELENGTH_M),
            "radial_samples": len(single_radial_axis_m),
            "angular_samples": 96,
            "normalization": "each profile divided by its first annular maximum",
            "fe_phase_source": "current compact L-BFGS-B FE, seed 172817509",
            "synthesis_wall_s": command["synthesis_wall_s"],
        })

SINGLE_RADIUS_AUDIT = pd.DataFrame(single_rows)
SINGLE_RADIUS_AUDIT.to_csv(TABLE_DIR / "single_sided_radius_reference_rmse.csv", index=False)
single_gs = SINGLE_RADIUS_AUDIT[SINGLE_RADIUS_AUDIT["method"].eq("GS")]
single_discrete_minimum = single_gs.loc[single_gs["radial_profile_rmse_to_deposited_fe"].idxmin()]
single_retained = single_gs[np.isclose(
    single_gs["radius_lambda"], SINGLE_RETAINED_RADIUS_LAMBDA
)].iloc[0]
single_nominal = single_gs[np.isclose(
    single_gs["radius_lambda"], NOMINAL_RADIUS_LAMBDA
)].iloc[0]
print(f"Single-sided FE first maximum: {single_radial_axis_m[single_fe_peak_index] / WAVELENGTH_M:.3f} lambda")
print(
    f"Retained R=0.45 lambda RMSE={single_retained['radial_profile_rmse_to_deposited_fe']:.8f}; "
    f"sampled minimum R={single_discrete_minimum['radius_lambda']:.2f} lambda, "
    f"RMSE={single_discrete_minimum['radial_profile_rmse_to_deposited_fe']:.8f}; "
    f"nominal R=1.40 lambda RMSE={single_nominal['radial_profile_rmse_to_deposited_fe']:.8f}"
)
